# Neo4j + GraphRAG Colab 教學版（疾病－用藥－飲食）

1. 連線到 **Neo4j AuraDB Free**
2. 建立「疾病－用藥－飲食」知識圖譜
3. 加入文字型衛教 `Chunk`
4. 為 `Chunk.text` 建立開源 embedding
5. 建立全文索引與向量索引
6. 示範 Graph Retrieval、Vector Retrieval、Full-text Retrieval
7. 把三種檢索結果組成 GraphRAG context
8. 丟給 LLM 生成答案

---

這份教材想讓學生看懂一件事：

> **GraphRAG 不是只有「把文件丟進向量資料庫」而已。**  
> 它也可以把「結構化知識圖譜」和「非結構化文字文件」一起拿來回答問題。

在這個範例中：

- **圖譜** 負責回答「病人有哪些疾病、吃哪些藥、哪些食物要避開」
- **文字 Chunk** 負責補充「衛教語句、飲食建議、用藥提醒」
- **LLM** 則負責把檢索到的證據整理成一段人類看得懂的回答

你可以把它理解成：

- Neo4j 圖譜：提供**關係推理**
- Embedding / 向量檢索：提供**語意相似**
- Full-text 檢索：提供**關鍵字命中**
- LLM：負責**最後表達**

## 你需要先準備

### Neo4j
- 一個 **Neo4j AuraDB Free** instance
- 你要知道：
  - `NEO4J_URI`
  - `NEO4J_USERNAME`
  - `NEO4J_PASSWORD`

### Groq
- 一個可用的 `GROQ_API_KEY`

### `.env` 檔案
請在 notebook 同層或工作目錄建立 `.env`，內容例如：

```env
NEO4J_URI=neo4j+s://xxxx.databases.neo4j.io
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your_password
GROQ_API_KEY=your_groq_api_key
```

In [ ]:
# ============================================================
# 參數與金鑰設定區（改用 .env）
# ------------------------------------------------------------
# 這裡改成從 .env 檔載入 Neo4j 與 Groq 的連線資訊。
#
# - load_dotenv() 會把 .env 內的 key/value 載入環境變數
# - 後面程式統一用 os.environ 讀取，不把帳密硬寫在程式裡
# ============================================================
import os
from dotenv import load_dotenv

# 載入 .env（預設找目前工作目錄）
load_dotenv()

# 檢查必要環境變數是否齊全
required_env = [
    "NEO4J_URI",
    "NEO4J_USERNAME",
    "NEO4J_PASSWORD",
    "GROQ_API_KEY",
]
missing = [k for k in required_env if not os.getenv(k)]
if missing:
    raise ValueError(f"缺少必要環境變數：{', '.join(missing)}，請確認 .env 檔案內容。")

# 生成模型改用 GROQ 的 LLaMA
# 這個模型負責最後把檢索到的證據整理成自然語言回答
GENERATION_MODEL = "llama-3.3-70b-versatile"

# embedding 改用本地開源模型 sentence-transformers
# 後面所有 Chunk 與使用者問題，都會被轉成同維度向量
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIMENSIONS = 384


NEO4J_URI··········
NEO4J_USERNAME··········
NEO4J_PASSWORD··········
GROQ_API_KEY··········


In [24]:
# ============================================================
# 核心物件初始化
# ------------------------------------------------------------
# 這一格會完成三件事：
# 1. 載入需要的 Python 套件
# 2. 建立 Neo4j driver，之後所有圖譜查詢都靠它
# 3. 建立 embedding model 與 Groq LLM client
#
# 這一格就是把整個 GraphRAG 系統會用到的三個引擎準備好：
# - 圖譜引擎：Neo4j
# - 向量引擎：SentenceTransformer
# - 生成引擎：Groq + LLaMA
# ============================================================
from neo4j import GraphDatabase
from groq import Groq
from sentence_transformers import SentenceTransformer
import pandas as pd
from tqdm.auto import tqdm
import json

# 從環境變數讀回剛剛設定的連線資訊
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
GROQ_API_KEY = os.environ["GROQ_API_KEY"]

# 建立 Neo4j driver
# 這裡可以把 driver 想成資料庫的「連線總控台」
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)

# 開源 embedding model：本地在 Colab 執行
# 作用：把文字轉成向量，供語意檢索使用
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

# GROQ client：專門拿來做 LLM 生成
# 作用：將 retrieval 結果整理成學生看得懂的回答
llm_client = Groq(api_key=GROQ_API_KEY)

def run_query(query, params=None, database=None):
    # --------------------------------------------------------
    # 統一的 Neo4j 查詢工具函式
    #
    # 「後面我們會一直寫 Cypher 查詢，但不想每次都重複開 session、
    #  執行 query、取回結果，所以先包成一個共用函式。」
    #
    # 參數說明：
    # - query：Cypher 查詢字串
    # - params：查詢參數，避免把變數直接硬寫進查詢中
    # - database：若有多個 database，可指定名稱
    #
    # 回傳格式：
    # - list[dict]，方便直接轉成 pandas DataFrame 觀察
    # --------------------------------------------------------
    params = params or {}
    with driver.session(database=database) as session:
        result = session.run(query, params)
        return [record.data() for record in result]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Step 1：清空資料庫並建立範例圖譜

這一步的目標，是先建立一個非常小、但足夠展示 GraphRAG 概念的醫療知識圖譜。

---


在這個示範中，我們先用幾種最容易理解的節點：

- `Patient` 病人
- `Disease` 疾病
- `Drug` 藥物
- `Food` 食物
- `Nutrient` 營養成分

然後用關係把它們串起來，例如：

- 病人有什麼疾病
- 病人正在吃什麼藥
- 某疾病應避免哪些食物
- 某藥物和哪些食物有交互作用

---

## 為什麼這一步很重要？

因為之後學生會看到：

- 純圖譜可以回答「已知關係」問題
- 但如果只有圖譜，通常缺少衛教語句與自然語言補充
- 所以後面才需要再把 `Chunk` 文件加進來


In [25]:
# ============================================================
# 建立示範知識圖譜
# ------------------------------------------------------------
# 這裡先把資料庫清空，避免重複執行 notebook 時資料越堆越多。
#
# ============================================================
run_query("MATCH (n) DETACH DELETE n")

# 這段 seed_cypher 會一次建立：
# - 1 位病人
# - 2 個疾病
# - 1 種藥物
# - 3 種食物
# - 2 個營養成分
#
# 然後再建立它們之間的關係。
#
seed_cypher = '''
CREATE (p:Patient {name:'王小明', age:58})
CREATE (d1:Disease {name:'糖尿病', category:'慢性病'})
CREATE (d2:Disease {name:'高血壓', category:'慢性病'})
CREATE (dr:Drug {name:'Metformin', warningLevel:'medium'})
CREATE (f1:Food {name:'高糖飲料', category:'飲料'})
CREATE (f2:Food {name:'葡萄柚', category:'水果'})
CREATE (f3:Food {name:'高鈉加工食品', category:'加工食品'})
CREATE (n1:Nutrient {name:'糖分'})
CREATE (n2:Nutrient {name:'鈉'})

CREATE (p)-[:HAS_DISEASE]->(d1)
CREATE (p)-[:HAS_DISEASE]->(d2)
CREATE (p)-[:TAKES]->(dr)

CREATE (dr)-[:TREATS]->(d1)
CREATE (d1)-[:SHOULD_AVOID]->(f1)
CREATE (d2)-[:SHOULD_AVOID]->(f3)

CREATE (dr)-[:INTERACTS_WITH {
  effect:'需注意服藥與飲食管理',
  evidenceLevel:'medium'
}]->(f2)

CREATE (f1)-[:CONTAINS]->(n1)
CREATE (f3)-[:CONTAINS]->(n2)
'''

# 將圖譜寫入 Neo4j
run_query(seed_cypher)

print("基礎圖譜建立完成")


基礎圖譜建立完成


In [26]:
# ============================================================
# 驗證圖譜是否建立成功
# ------------------------------------------------------------
# 這一格用一個簡單 Cypher 查詢，把病人相關的主要資訊抓出來。
#
# 1. 先找到指定病人
# 2. 再沿著關係往外抓疾病、藥物、應避免的食物
# 3. 用 collect() 把多個結果整理成清單
#
# ============================================================
query = '''
MATCH (p:Patient {name:'王小明'})-[:HAS_DISEASE]->(d:Disease)
OPTIONAL MATCH (p)-[:TAKES]->(dr:Drug)
OPTIONAL MATCH (d)-[:SHOULD_AVOID]->(f:Food)
RETURN p.name AS patient,
       collect(DISTINCT d.name) AS diseases,
       collect(DISTINCT dr.name) AS drugs,
       collect(DISTINCT f.name) AS avoid_foods
'''

# 用 DataFrame 顯示，教學時比較直觀
pd.DataFrame(run_query(query))


,patient,diseases,drugs,avoid_foods
0,王小明,"[糖尿病, 高血壓]",[Metformin],"[高糖飲料, 高鈉加工食品]"


## Step 2：加入文字型 Chunk（衛教文字）

到這裡為止，我們只有「圖譜中的關係」，還沒有「自然語言說明」。

所以這一步要補進第二種資料來源：**文字文件 Chunk**。

---

這裡一定要理解一個核心差異：

### 圖譜擅長
- 誰跟誰有關係
- 哪些節點彼此相連
- 結構化查詢

### 文件 Chunk 擅長
- 用自然語言寫下衛教說明
- 補充圖譜沒有明確列出的語句
- 讓最後回答更像「人會說的話」

所以接下來會出現一個典型 GraphRAG 設計：

> **圖譜給骨架，文件給語意內容。**

---

## 為什麼叫 Chunk？

因為在 RAG 裡，長文件通常會先切成一小段一小段，再做 embedding 與檢索。  
這裡為了教學簡化，直接手動建立幾個小段文字。


In [27]:
# ============================================================
# 建立文字型 Chunk
# ------------------------------------------------------------
# 這裡的 chunks 可以把它理解成「小型衛教知識片段」。
#
# 每個 chunk 都包含：
# - chunk_id：唯一識別碼
# - title：標題，方便辨識
# - text：真正要拿來檢索與回答的文字內容
#
# 正式專案中，這些 chunk 常常不是手工寫的，
# 而是從 PDF、網頁、SOP、衛教單、病歷摘要等來源切分出來的。
# ============================================================
chunks = [
    {
        "chunk_id": "c1",
        "title": "糖尿病飲食衛教",
        "text": "糖尿病患者應避免高糖飲料、含糖手搖飲與精緻甜食，晚餐建議以低糖、均衡飲食為主。"
    },
    {
        "chunk_id": "c2",
        "title": "Metformin 用藥說明",
        "text": "Metformin 常用於第二型糖尿病管理，服藥期間應留意規律飲食與血糖控制。"
    },
    {
        "chunk_id": "c3",
        "title": "高血壓飲食建議",
        "text": "高血壓患者應減少高鈉加工食品攝取，避免過鹹飲食，並優先選擇清淡烹調。"
    },
    {
        "chunk_id": "c4",
        "title": "葡萄柚注意事項",
        "text": "部分藥物與葡萄柚可能有交互作用，服藥前應確認是否需要避免同時食用。"
    },
    {
        "chunk_id": "c5",
        "title": "晚餐設計原則",
        "text": "對於同時有糖尿病與高血壓的患者，晚餐建議同時兼顧低糖與低鈉原則。"
    },
    {
        "chunk_id": "c6",
        "title": "病人教育摘要",
        "text": "若患者同時服用降糖藥並有慢性病，飲食建議應同時參考疾病限制與藥物交互作用。"
    }
]

# 先把 chunks 轉成表格，讓學生快速瀏覽資料內容
pd.DataFrame(chunks)


,chunk_id,title,text
0,c1,糖尿病飲食衛教,糖尿病患者應避免高糖飲料、含糖手搖飲與精緻甜食，晚餐建議以低糖、均衡飲食為主。
1,c2,Metformin 用藥說明,Metformin 常用於第二型糖尿病管理，服藥期間應留意規律飲食與血糖控制。
2,c3,高血壓飲食建議,高血壓患者應減少高鈉加工食品攝取，避免過鹹飲食，並優先選擇清淡烹調。
3,c4,葡萄柚注意事項,部分藥物與葡萄柚可能有交互作用，服藥前應確認是否需要避免同時食用。
4,c5,晚餐設計原則,對於同時有糖尿病與高血壓的患者，晚餐建議同時兼顧低糖與低鈉原則。
5,c6,病人教育摘要,若患者同時服用降糖藥並有慢性病，飲食建議應同時參考疾病限制與藥物交互作用。


In [28]:
# ============================================================
# 將 Chunk 寫入圖譜，並建立 Chunk 與實體之間的連結
# ------------------------------------------------------------
# 這一格做兩件事：
#
# 第 1 件事：把每段文字存成 Chunk 節點
# 第 2 件事：把 Chunk 與 Disease / Drug / Food 等節點連起來
#
# 這個設計非常重要，因為它讓我們的文件不再只是孤立文字，
# 而是可以「掛回知識圖譜」。
#
# ============================================================

# 先建立 Chunk 節點，將 title 與 text 存進 Neo4j
for c in chunks:
    run_query(
        '''
        MERGE (ch:Chunk {chunk_id: $chunk_id})
        SET ch.title = $title,
            ch.text = $text
        ''',
        c
    )

# link_queries 用來描述：
# 哪一個 Chunk 提到了哪個實體（Disease / Drug / Food）
#
# 例如：
# - c1 提到 糖尿病 與 高糖飲料
# - c2 提到 Metformin 與 糖尿病
# 這樣後面就能從文件回連到圖譜實體
link_queries = [
    ("c1", "Disease", "糖尿病"),
    ("c1", "Food", "高糖飲料"),
    ("c2", "Drug", "Metformin"),
    ("c2", "Disease", "糖尿病"),
    ("c3", "Disease", "高血壓"),
    ("c3", "Food", "高鈉加工食品"),
    ("c4", "Food", "葡萄柚"),
    ("c4", "Drug", "Metformin"),
    ("c5", "Disease", "糖尿病"),
    ("c5", "Disease", "高血壓"),
    ("c6", "Drug", "Metformin"),
    ("c6", "Disease", "糖尿病"),
]

# 逐筆建立 (Chunk)-[:MENTIONS]->(實體) 關係
for chunk_id, label, name in link_queries:
    q = f'''
    MATCH (ch:Chunk {{chunk_id: $chunk_id}})
    MATCH (n:{label} {{name: $name}})
    MERGE (ch)-[:MENTIONS]->(n)
    '''
    run_query(q, {"chunk_id": chunk_id, "name": name})

print("Chunk 與圖譜實體的連結建立完成")


ERROR:neo4j.io:[#B3C2]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-mt-3b814dd7cab7-2-0029.production-orch-0068.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.171.25', 7687))): OSError('No data')


SessionExpired: Failed to read from defunct connection IPv4Address(('p-mt-3b814dd7cab7-2-0029.production-orch-0068.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.171.25', 7687)))

In [8]:
# ============================================================
# 檢查 Chunk 與實體的連結是否建立成功
# ------------------------------------------------------------
# 這一格的目的，是確認每個 Chunk 真的有連到它提到的實體。
#
# - c1 連到糖尿病、高糖飲料
# - c2 連到 Metformin、糖尿病
# - c5 同時連到糖尿病與高血壓
#
# 這表示我們的文件資料已經「嵌進圖譜結構」中了。
# ============================================================
query = '''
MATCH (ch:Chunk)-[:MENTIONS]->(n)
RETURN ch.chunk_id, ch.title, labels(n) AS labels, n.name
ORDER BY ch.chunk_id
'''
pd.DataFrame(run_query(query))


,ch.chunk_id,ch.title,labels,n.name
0,c1,糖尿病飲食衛教,[Disease],糖尿病
1,c1,糖尿病飲食衛教,[Food],高糖飲料
2,c2,Metformin 用藥說明,[Disease],糖尿病
3,c2,Metformin 用藥說明,[Drug],Metformin
4,c3,高血壓飲食建議,[Disease],高血壓
5,c3,高血壓飲食建議,[Food],高鈉加工食品
6,c4,葡萄柚注意事項,[Drug],Metformin
7,c4,葡萄柚注意事項,[Food],葡萄柚
8,c5,晚餐設計原則,[Disease],糖尿病
9,c5,晚餐設計原則,[Disease],高血壓


## Step 3：為 Chunk 建立 embedding，並寫回 Neo4j



In [9]:
# ============================================================
# 定義 embedding 函式
# ------------------------------------------------------------
# get_embedding() 的工作非常單純：
# 輸入一段文字，輸出一個向量 list。
#
# normalize_embeddings=True 的意思，是先把向量正規化，
# 這樣之後用 cosine similarity 做相似度比較會更穩定。
#
# ============================================================
def get_embedding(text: str):
    emb = embedding_model.encode(
        text,
        normalize_embeddings=True
    )
    return emb.tolist()

# 先做一次測試，確認 embedding 功能正常
test_emb = get_embedding("糖尿病患者應避免高糖飲料")

# 顯示向量長度與前 5 個值
# 這裡主要是讓學生知道：文字真的被轉成固定維度的向量了
len(test_emb), test_emb[:5]


(384,
 [0.009256032295525074,
  0.10733082890510559,
  0.03690330684185028,
  0.0375051349401474,
  -0.047962725162506104])

In [10]:
# ============================================================
# 將每個 Chunk 的 embedding 寫回 Neo4j
# ------------------------------------------------------------
# 流程如下：
# 1. 逐筆讀取 chunks
# 2. 把每段 text 轉成 embedding
# 3. 寫回對應的 Chunk 節點屬性 ch.embedding
#
# ============================================================
for c in tqdm(chunks):
    emb = get_embedding(c["text"])
    run_query(
        '''
        MATCH (ch:Chunk {chunk_id: $chunk_id})
        SET ch.embedding = $embedding
        ''',
        {"chunk_id": c["chunk_id"], "embedding": emb}
    )

print("Chunk embedding 寫回完成")


  0%|          | 0/6 [00:00<?, ?it/s]

Chunk embedding 寫回完成


## Step 4：建立全文索引與向量索引

這一步是在幫 Neo4j 準備兩種檢索機制：

1. **Full-text index**：用關鍵字找文字
2. **Vector index**：用語意相似度找文字

In [12]:
# ============================================================
# 建立全文索引與向量索引
# ------------------------------------------------------------
# Full-text index：
#   讓我們可以對 Chunk.title 與 Chunk.text 做關鍵字檢索
#
# Vector index：
#   讓我們可以對 Chunk.embedding 做向量相似度檢索
#
# ============================================================

# 建立全文索引
fulltext_index_cypher = '''
CREATE FULLTEXT INDEX chunkFulltext IF NOT EXISTS
FOR (c:Chunk) ON EACH [c.title, c.text];
'''
run_query(fulltext_index_cypher)

# 建立向量索引
vector_index_cypher = f'''
CREATE VECTOR INDEX chunkEmbeddingIndex IF NOT EXISTS
FOR (c:Chunk) ON c.embedding
OPTIONS {{
  indexConfig: {{
    `vector.dimensions`: {EMBEDDING_DIMENSIONS},
    `vector.similarity_function`: 'cosine'
  }}
}};
'''
run_query(vector_index_cypher)

print("索引建立指令已送出")

索引建立指令已送出


In [13]:
# ============================================================
# 檢查索引狀態
# ------------------------------------------------------------
# 這一格目的：
# - 確認索引有沒有建立成功
# - 觀察索引名稱、型別、狀態、作用欄位
#
# 老師可提醒學生：
# 如果索引還沒 ready，就可能導致後面查詢失敗或沒有結果。
# ============================================================
pd.DataFrame(run_query(
    "SHOW INDEXES YIELD name, type, state, labelsOrTypes, properties RETURN *"
))


,name,type,state,labelsOrTypes,properties
0,chunkEmbeddingIndex,VECTOR,ONLINE,[Chunk],[embedding]
1,chunkFulltext,FULLTEXT,ONLINE,[Chunk],"[title, text]"
2,index_1b9dcc97,LOOKUP,ONLINE,None,None
3,index_460996c0,LOOKUP,ONLINE,None,None


## Step 5：做三種 retrieval

接下來是整份教材最關鍵的地方之一。

我們會把同一個問題，分別交給三種 retrieval 方法：

1. **Graph Retrieval**
2. **Vector Retrieval**
3. **Full-text Retrieval**

---
## 三種 retrieval 的角色分工

### 1. Graph Retrieval
找回圖譜中的**已知關係**
- 病人有什麼病
- 正在吃什麼藥
- 疾病應避免什麼食物
- 藥物可能和什麼食物有交互作用

### 2. Vector Retrieval
找回語意相近的文字 Chunk
- 不一定字面完全一樣
- 但語意上很接近問題需求

### 3. Full-text Retrieval
找回關鍵字相符的文字 Chunk
- 對專有名詞常很有用
- 例如藥名、疾病名、飲食名詞


In [14]:
# ============================================================
# 1) Graph Retrieval：從圖譜中找「已知關係」
# ------------------------------------------------------------
# 這個函式只看圖譜，不看文件文字。
#
# 它會回傳：
# - 病人名稱
# - 疾病列表
# - 用藥列表
# - 疾病應避免食物
# - 藥物交互作用食物
#
# ============================================================
def graph_retrieval(patient_name: str):
    query = '''
    MATCH (p:Patient {name: $patient_name})-[:HAS_DISEASE]->(d:Disease)
    OPTIONAL MATCH (p)-[:TAKES]->(dr:Drug)
    OPTIONAL MATCH (d)-[:SHOULD_AVOID]->(f1:Food)
    OPTIONAL MATCH (dr)-[r:INTERACTS_WITH]->(f2:Food)
    RETURN p.name AS patient,
           collect(DISTINCT d.name) AS diseases,
           collect(DISTINCT dr.name) AS drugs,
           collect(DISTINCT f1.name) AS disease_avoid_foods,
           collect(DISTINCT f2.name) AS drug_interaction_foods
    '''
    rows = run_query(query, {"patient_name": patient_name})
    return rows[0] if rows else {}

# 試查：看圖譜本身能回多少資訊
graph_retrieval("王小明")


{'patient': '王小明',
 'diseases': ['糖尿病', '高血壓'],
 'drugs': ['Metformin'],
 'disease_avoid_foods': ['高糖飲料', '高鈉加工食品'],
 'drug_interaction_foods': ['葡萄柚']}

In [15]:
# ============================================================
# 2) Vector Retrieval：從 Chunk 中找「語意最接近」的內容
# ------------------------------------------------------------
# 步驟：
# 1. 先把使用者問題轉成 embedding
# 2. 呼叫 Neo4j 的向量索引查詢
# 3. 找出 Top-k 最相近的 Chunk
#
# ============================================================
def vector_retrieval(question: str, k: int = 3):
    # 將問題文字轉成向量
    q_emb = get_embedding(question)

    query = '''
    CALL db.index.vector.queryNodes('chunkEmbeddingIndex', $k, $embedding)
    YIELD node, score
    RETURN node.chunk_id AS chunk_id,
           node.title AS title,
           node.text AS text,
           score
    ORDER BY score DESC
    '''
    return run_query(query, {"k": k, "embedding": q_emb})

# 示範向量檢索結果
pd.DataFrame(vector_retrieval("糖尿病患者服用 Metformin，晚餐應避免什麼？", k=3))


,chunk_id,title,text,score
0,c2,Metformin 用藥說明,Metformin 常用於第二型糖尿病管理，服藥期間應留意規律飲食與血糖控制。,0.926180
1,c4,葡萄柚注意事項,部分藥物與葡萄柚可能有交互作用，服藥前應確認是否需要避免同時食用。,0.762261
2,c1,糖尿病飲食衛教,糖尿病患者應避免高糖飲料、含糖手搖飲與精緻甜食，晚餐建議以低糖、均衡飲食為主。,0.756832


In [16]:
# ============================================================
# 3) Full-text Retrieval：從 Chunk 中找「關鍵字相符」的內容
# ------------------------------------------------------------
# 這個方法與向量檢索不同：
# - 向量檢索偏重語意
# - 全文檢索偏重字面命中
#
# ============================================================
def fulltext_retrieval(query_text: str, k: int = 3):
    query = '''
    CALL db.index.fulltext.queryNodes('chunkFulltext', $query_text)
    YIELD node, score
    RETURN node.chunk_id AS chunk_id,
           node.title AS title,
           node.text AS text,
           score
    ORDER BY score DESC
    LIMIT $k
    '''
    return run_query(query, {"query_text": query_text, "k": k})

# 示範全文檢索結果
pd.DataFrame(fulltext_retrieval("糖尿病 Metformin 晚餐 食物", k=3))


,chunk_id,title,text,score
0,c1,糖尿病飲食衛教,糖尿病患者應避免高糖飲料、含糖手搖飲與精緻甜食，晚餐建議以低糖、均衡飲食為主。,4.112123
1,c5,晚餐設計原則,對於同時有糖尿病與高血壓的患者，晚餐建議同時兼顧低糖與低鈉原則。,3.195382
2,c2,Metformin 用藥說明,Metformin 常用於第二型糖尿病管理，服藥期間應留意規律飲食與血糖控制。,2.443272


## Step 6：把三種 retrieval 組成 GraphRAG context

現在我們終於要把 GraphRAG 的核心拼起來了。

前面三種 retrieval 各自能取回不同類型的證據，  
這一步則是把它們整合成一個 context，準備交給 LLM。


In [17]:
# ============================================================
# 建立 GraphRAG context
# ------------------------------------------------------------
# 這個函式的角色，就是把三種 retrieval 的結果打包在一起。
#
# 最後回傳的 context 會包含：
# - 原始問題
# - graph_data：圖譜結果
# - vector_hits：向量檢索結果
# - fulltext_hits：全文檢索結果
#
# ============================================================
def build_graphrag_context(question: str, patient_name: str = "王小明", k_vector: int = 3, k_fulltext: int = 3):
    # 從圖譜取回結構化關係
    graph_data = graph_retrieval(patient_name)

    # 從向量索引取回語意相近文件
    vector_hits = vector_retrieval(question, k=k_vector)

    # 從全文索引取回關鍵字相近文件
    fulltext_hits = fulltext_retrieval(question, k=k_fulltext)

    # 組成統一 context，後面會直接餵給 LLM
    context = {
        "question": question,
        "graph_data": graph_data,
        "vector_hits": vector_hits,
        "fulltext_hits": fulltext_hits,
    }
    return context

# 試著建立一份完整的 GraphRAG context
context = build_graphrag_context("王小明有糖尿病又有高血壓，正在吃 Metformin，晚餐要注意什麼？")
context


{'question': '王小明有糖尿病又有高血壓，正在吃 Metformin，晚餐要注意什麼？',
 'graph_data': {'patient': '王小明',
  'diseases': ['糖尿病', '高血壓'],
  'drugs': ['Metformin'],
  'disease_avoid_foods': ['高糖飲料', '高鈉加工食品'],
  'drug_interaction_foods': ['葡萄柚']},
 'vector_hits': [{'chunk_id': 'c3',
   'title': '高血壓飲食建議',
   'text': '高血壓患者應減少高鈉加工食品攝取，避免過鹹飲食，並優先選擇清淡烹調。',
   'score': 0.8574283123016357},
  {'chunk_id': 'c2',
   'title': 'Metformin 用藥說明',
   'text': 'Metformin 常用於第二型糖尿病管理，服藥期間應留意規律飲食與血糖控制。',
   'score': 0.8408699035644531},
  {'chunk_id': 'c1',
   'title': '糖尿病飲食衛教',
   'text': '糖尿病患者應避免高糖飲料、含糖手搖飲與精緻甜食，晚餐建議以低糖、均衡飲食為主。',
   'score': 0.839658260345459}],
 'fulltext_hits': [{'chunk_id': 'c5',
   'title': '晚餐設計原則',
   'text': '對於同時有糖尿病與高血壓的患者，晚餐建議同時兼顧低糖與低鈉原則。',
   'score': 4.961688995361328},
  {'chunk_id': 'c2',
   'title': 'Metformin 用藥說明',
   'text': 'Metformin 常用於第二型糖尿病管理，服藥期間應留意規律飲食與血糖控制。',
   'score': 4.160144805908203},
  {'chunk_id': 'c1',
   'title': '糖尿病飲食衛教',
   'text': '糖尿病患者應避免高糖飲料、含糖手搖飲與精緻甜食，晚餐建議以低糖、均

In [18]:
# ============================================================
# 將 context 轉成給 LLM 的 prompt
# ------------------------------------------------------------
# 這一步非常重要，因為 retrieval 做得再好，
# 如果 prompt 組得不好，模型還是可能回答不理想。
#
# 這裡的設計理念是：
# 1. 明確限制模型只能依據提供證據回答
# 2. 把圖譜結果與文件結果分開呈現
# 3. 指定回答格式，讓輸出更穩定
# ============================================================
def format_context_for_llm(context: dict) -> str:
    graph_data = context["graph_data"]
    vector_hits = context["vector_hits"]
    fulltext_hits = context["fulltext_hits"]

    # 把向量檢索結果整理成一段可閱讀文字
    vector_text = "\n\n".join(
        [f"[Vector-{i+1}] {hit['title']}\n{hit['text']}" for i, hit in enumerate(vector_hits)]
    )

    # 把全文檢索結果整理成一段可閱讀文字
    fulltext_text = "\n\n".join(
        [f"[FullText-{i+1}] {hit['title']}\n{hit['text']}" for i, hit in enumerate(fulltext_hits)]
    )

    # 回傳完整 prompt
    return f'''
你是一位醫療知識助理。請只根據以下提供的圖譜結果與文字證據回答，不要自行捏造未提供的醫療事實。
如果資料不足，請明確說「目前圖譜與文件證據不足」。

【使用者問題】
{context["question"]}

【圖譜結果】
病人：{graph_data.get("patient")}
疾病：{graph_data.get("diseases")}
目前用藥：{graph_data.get("drugs")}
疾病應避免食物：{graph_data.get("disease_avoid_foods")}
藥物交互作用食物：{graph_data.get("drug_interaction_foods")}

【語意相似文件（Vector Retrieval）】
{vector_text}

【關鍵字文件（Full-text Retrieval）】
{fulltext_text}

【回答要求】
1. 用繁體中文回答
2. 先給 3 點重點建議
3. 再用短段落說明理由
4. 最後附上「本次回答依據」：
   - 圖譜依據
   - 文件依據
5. 不要把這當成正式醫囑，請加上一句提醒使用者仍應諮詢專業醫療人員
'''.strip()

# 先把 prompt 印出來看看，讓學生知道 LLM 實際看到的是什麼
llm_prompt = format_context_for_llm(context)
print(llm_prompt[:2000])


你是一位醫療知識助理。請只根據以下提供的圖譜結果與文字證據回答，不要自行捏造未提供的醫療事實。
如果資料不足，請明確說「目前圖譜與文件證據不足」。

【使用者問題】
王小明有糖尿病又有高血壓，正在吃 Metformin，晚餐要注意什麼？

【圖譜結果】
病人：王小明
疾病：['糖尿病', '高血壓']
目前用藥：['Metformin']
疾病應避免食物：['高糖飲料', '高鈉加工食品']
藥物交互作用食物：['葡萄柚']

【語意相似文件（Vector Retrieval）】
[Vector-1] 高血壓飲食建議
高血壓患者應減少高鈉加工食品攝取，避免過鹹飲食，並優先選擇清淡烹調。

[Vector-2] Metformin 用藥說明
Metformin 常用於第二型糖尿病管理，服藥期間應留意規律飲食與血糖控制。

[Vector-3] 糖尿病飲食衛教
糖尿病患者應避免高糖飲料、含糖手搖飲與精緻甜食，晚餐建議以低糖、均衡飲食為主。

【關鍵字文件（Full-text Retrieval）】
[FullText-1] 晚餐設計原則
對於同時有糖尿病與高血壓的患者，晚餐建議同時兼顧低糖與低鈉原則。

[FullText-2] Metformin 用藥說明
Metformin 常用於第二型糖尿病管理，服藥期間應留意規律飲食與血糖控制。

[FullText-3] 糖尿病飲食衛教
糖尿病患者應避免高糖飲料、含糖手搖飲與精緻甜食，晚餐建議以低糖、均衡飲食為主。

【回答要求】
1. 用繁體中文回答
2. 先給 3 點重點建議
3. 再用短段落說明理由
4. 最後附上「本次回答依據」：
   - 圖譜依據
   - 文件依據
5. 不要把這當成正式醫囑，請加上一句提醒使用者仍應諮詢專業醫療人員


In [19]:
# ============================================================
# 呼叫 LLM 生成最終答案
# ------------------------------------------------------------
# 到這裡才真正進入「生成」階段。
#
# temperature=0.2：
# - 讓模型回答更穩定
# - 降低太發散、太創作式的風險
# - 對醫療類示範較合適
# ============================================================
def generate_answer_with_llm(prompt: str, model: str = GENERATION_MODEL):
    response = llm_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "你是嚴謹、清楚、保守的醫療知識整理助手。"},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

# 生成答案並印出
answer = generate_answer_with_llm(llm_prompt)
print(answer)


根據提供的圖譜結果與文件證據，以下是三點重點建議：

1. 避免高糖飲料和高鈉加工食品。
2. 優先選擇清淡烹調和低糖、均衡飲食。
3. 注意避免食用葡萄柚，以免與 Metformin 產生藥物交互作用。

晚餐是王小明一天中重要的一餐，需要特別注意飲食內容。由於王小明同時患有糖尿病和高血壓，需要兼顧低糖和低鈉原則。根據文件證據，高血壓患者應減少高鈉加工食品攝取，避免過鹹飲食；糖尿病患者應避免高糖飲料、含糖手搖飲與精緻甜食。同時，王小明正在服用 Metformin，需要注意規律飲食與血糖控制，並避免食用可能產生藥物交互作用的食物，如葡萄柯。

本次回答依據：
- 圖譜依據：王小明的疾病、目前用藥、疾病應避免食物和藥物交互作用食物。
- 文件依據：高血壓飲食建議、Metformin 用藥說明、糖尿病飲食衛教、晚餐設計原則等。

請注意，這些建議不應當作正式醫囑，使用者仍應諮詢專業醫療人員以獲得個性化的醫療建議。


## Step 7：把流程包成可互動問答函式

到目前為止，我們已經有所有零件了：

- 圖譜查詢
- 向量檢索
- 全文檢索
- Prompt 組裝
- LLM 回答

這一步要做的，就是把整條流程包成一個可以重複呼叫的函式。



In [20]:
# ============================================================
# 包成完整的問答函式
# ------------------------------------------------------------
# ask_medical_graphrag() 是整份 notebook 的主流程函式。
#
# 它會依序做：
# 1. 建立 context
# 2. 轉成 prompt
# 3. 呼叫 LLM 生成答案
# 4. 回傳完整結果（包含 context 與 answer）
#
# ============================================================
def ask_medical_graphrag(question: str, patient_name: str = "王小明"):
    context = build_graphrag_context(question=question, patient_name=patient_name)
    prompt = format_context_for_llm(context)
    answer = generate_answer_with_llm(prompt)
    return {
        "question": question,
        "context": context,
        "answer": answer,
    }

# 實際發問測試
result = ask_medical_graphrag("王小明有糖尿病又有高血壓，正在吃 Metformin，晚餐建議怎麼安排？")
print(result["answer"])


王小明晚餐建議的三點重點如下：
1. 避免高糖飲料和高鈉加工食品。
2. 優先選擇清淡烹調和低糖、均衡飲食。
3. 注意避免與Metformin有交互作用的食物，如葡萄柚。

這些建議的理由是基於王小明同時患有糖尿病和高血壓，需要兼顧低糖和低鈉的飲食原則。根據提供的圖譜和文件，高血壓患者應減少高鈉加工食品攝取，避免過鹹飲食，而糖尿病患者應避免高糖飲料和精緻甜食。同時，Metformin的用藥說明也提醒患者應留意規律飲食與血糖控制，並避免與藥物有交互作用的食物。

本次回答依據：
- 圖譜依據：王小明的疾病、目前用藥、疾病應避免食物和藥物交互作用食物。
- 文件依據：高血壓飲食建議、Metformin用藥說明、糖尿病飲食衛教、晚餐設計原則等相關文件。

請注意，這些建議並不取代專業醫療人員的意見，使用者仍應諮詢專業醫療人員以獲得最適合的個人化建議。


## Step 8：比較三種模式的差異

### 純 Graph
優點：
- 結構清楚
- 關係明確
- 不容易亂抓

限制：
- 不會自動補出自然語言建議
- 若圖譜沒建進去，就查不到

### 純 Vector
優點：
- 能抓語意相近內容
- 對自然語言提問很友善

限制：
- 可能抓到語意像、但不是最關鍵的段落

### 純 Full-text
優點：
- 關鍵字命中很直接
- 對專有名詞敏感

限制：
- 使用者換個說法時，可能抓不到



In [21]:
# ============================================================
# 比較三種 retrieval 模式
# ------------------------------------------------------------
# 這一格會把同一個問題分別用三種方式查看結果：
# 1. 純 Graph
# 2. 純 Vector
# 3. 純 Full-text
#
# ============================================================
question = "王小明有糖尿病又有高血壓，正在吃 Metformin，晚餐建議怎麼安排？"

print("=== 純 Graph 結果 ===")
print(json.dumps(graph_retrieval("王小明"), ensure_ascii=False, indent=2))

print("\n=== 純 Vector Top-3 ===")
print(pd.DataFrame(vector_retrieval(question, k=3))[["title", "score"]])

print("\n=== Full-text Top-3 ===")
print(pd.DataFrame(fulltext_retrieval("糖尿病 Metformin 晚餐 高血壓", k=3))[["title", "score"]])


=== 純 Graph 結果 ===
{
  "patient": "王小明",
  "diseases": [
    "糖尿病",
    "高血壓"
  ],
  "drugs": [
    "Metformin"
  ],
  "disease_avoid_foods": [
    "高糖飲料",
    "高鈉加工食品"
  ],
  "drug_interaction_foods": [
    "葡萄柚"
  ]
}

=== 純 Vector Top-3 ===
            title     score
0         高血壓飲食建議  0.849322
1  Metformin 用藥說明  0.838696
2         糖尿病飲食衛教  0.832142

=== Full-text Top-3 ===
     title     score
0   晚餐設計原則  4.317689
1  糖尿病飲食衛教  3.819103
2  高血壓飲食建議  3.239561


## 總結

這份 Colab 示範了三個層次：

### 第 1 層：Neo4j 圖查詢
你可以用 Cypher 找出病人、疾病、藥物、食物之間的路徑

### 第 2 層：圖 + 文件
你可以把衛教文字 Chunk 加進來，並且把文件和圖譜實體連起來

### 第 3 層：GraphRAG 問答
你可以把圖譜結果、向量檢索結果、全文檢索結果組合後交給 LLM 生成回答